In [1]:
import os
import json
import numpy as np
import pandas as pd
import zarr
from scipy.ndimage import gaussian_filter, maximum_filter
from scipy.optimize import linear_sum_assignment

DATASET_PATH = '/kaggle/input/competitions/biohub-cell-tracking-during-development'
TEST_DIR = f'{DATASET_PATH}/test'

SCALE = np.array([1.625, 0.40625, 0.40625], dtype=np.float64)  # Z, Y, X um/voxel

# ---- detection: DoG blob detector (recovers dim nuclei, better than plain Otsu) ----

def _ball_footprint(radius_um, eff_spacing):
    rad_vox = np.maximum(1, np.round(radius_um / eff_spacing).astype(int))
    zz, yy, xx = np.ogrid[-rad_vox[0]:rad_vox[0]+1, -rad_vox[1]:rad_vox[1]+1, -rad_vox[2]:rad_vox[2]+1]
    d = (zz*eff_spacing[0])**2 + (yy*eff_spacing[1])**2 + (xx*eff_spacing[2])**2
    return d <= radius_um**2

def detect_blobs(vol, xy_downsample=4, dog_scales=None, rel_threshold=0.045,
                  abs_percentile=50.0, min_distance_um=4.0, max_peaks=40000):
    vf = vol.astype(np.float32)
    ds = vf[:, ::xy_downsample, ::xy_downsample]
    eff = np.array([SCALE[0], SCALE[1]*xy_downsample, SCALE[2]*xy_downsample])

    lo, hi = np.percentile(ds, [1.0, 99.7])
    if hi <= lo:
        hi = lo + 1.0
    norm = np.clip((ds - lo) / (hi - lo), 0, None)

    if dog_scales is None:
        dog_scales = [[1.5, 4.0], [2.2, 5.5]]
    dog = None
    for s_um, l_um in dog_scales:
        resp = gaussian_filter(norm, sigma=s_um/eff) - gaussian_filter(norm, sigma=l_um/eff)
        dog = resp if dog is None else np.maximum(dog, resp)

    footprint = _ball_footprint(min_distance_um, eff)
    mx = maximum_filter(dog, footprint=footprint, mode='nearest')
    abs_thr = np.percentile(norm, abs_percentile)
    peaks = (dog == mx) & (dog >= rel_threshold) & (norm >= abs_thr)
    coords = np.argwhere(peaks)
    if coords.size == 0:
        return np.zeros((0, 3), dtype=np.float64)

    vals = dog[peaks]
    order = np.argsort(vals)[::-1]
    coords = coords[order]
    if max_peaks is not None and len(coords) > max_peaks:
        coords = coords[:max_peaks]

    out = coords.astype(np.float64)
    out[:, 1] *= xy_downsample
    out[:, 2] *= xy_downsample
    return out

def refine_centroids(vol, coords, win=(3, 9, 9)):
    if len(coords) == 0:
        return coords
    Z, Y, X = vol.shape
    out = coords.copy().astype(np.float64)
    wz, wy, wx = win
    for i, (z, y, x) in enumerate(coords):
        z, y, x = int(round(z)), int(round(y)), int(round(x))
        z0, z1 = max(0, z-wz), min(Z, z+wz+1)
        y0, y1 = max(0, y-wy), min(Y, y+wy+1)
        x0, x1 = max(0, x-wx), min(X, x+wx+1)
        patch = vol[z0:z1, y0:y1, x0:x1].astype(np.float64)
        s = patch.sum()
        if s <= 0:
            continue
        zz = np.arange(z0, z1)[:, None, None]
        yy = np.arange(y0, y1)[None, :, None]
        xx = np.arange(x0, x1)[None, None, :]
        out[i, 0] = (patch*zz).sum()/s
        out[i, 1] = (patch*yy).sum()/s
        out[i, 2] = (patch*xx).sum()/s
    return out

# ---- linking: two-pass Hungarian (tight gate, then loose gate for leftovers) ----

def link_twopass(frames, tight_um=6.0, loose_um=8.0):
    node_ids, node_t, node_z, node_y, node_x, frame_ids = [], [], [], [], [], []
    nid = 1
    for t, coords in enumerate(frames):
        ids = []
        for z, y, x in coords:
            node_ids.append(nid); node_t.append(t); node_z.append(z); node_y.append(y); node_x.append(x)
            ids.append(nid); nid += 1
        frame_ids.append(ids)

    def _hun(P, C, pi, ci, gate):
        if len(pi) == 0 or len(ci) == 0:
            return []
        BIG = 1e9
        D = np.sqrt(((P[pi][:, None] - C[ci][None])**2).sum(2))
        cost = np.where(D > gate, BIG, D)
        ri, rc = linear_sum_assignment(cost)
        return [(int(pi[r]), int(ci[c])) for r, c in zip(ri, rc) if cost[r, c] < BIG]

    edges = []
    for t in range(len(frames) - 1):
        a, b = np.asarray(frames[t], float).reshape(-1, 3), np.asarray(frames[t+1], float).reshape(-1, 3)
        if len(a) == 0 or len(b) == 0:
            continue
        P, C = a * SCALE, b * SCALE
        N, M = len(P), len(C)
        links = _hun(P, C, np.arange(N), np.arange(M), tight_um)
        up = {p for p, _ in links}; uc = {c for _, c in links}
        fp = np.array([i for i in range(N) if i not in up], int)
        fc = np.array([j for j in range(M) if j not in uc], int)
        links += _hun(P, C, fp, fc, loose_um)
        for p, c in links:
            edges.append((frame_ids[t][p], frame_ids[t+1][c]))

    return (np.array(node_ids), np.array(node_t), np.array(node_z), np.array(node_y),
            np.array(node_x), np.array(edges, dtype=np.int64).reshape(-1, 2))

def filter_short_tracks(node_ids, edges, min_len=4):
    if len(edges) == 0:
        return node_ids, edges
    parent = {int(n): int(n) for n in node_ids}
    def find(a):
        while parent[a] != a:
            parent[a] = parent[parent[a]]; a = parent[a]
        return a
    for s, t in edges:
        ra, rb = find(int(s)), find(int(t))
        if ra != rb:
            parent[ra] = rb
    comp = {}
    for n in node_ids:
        comp.setdefault(find(int(n)), []).append(int(n))
    keep = set()
    for members in comp.values():
        if len(members) >= min_len:
            keep.update(members)
    keep_edges = np.array([(s, t) for s, t in edges if int(s) in keep and int(t) in keep],
                           dtype=np.int64).reshape(-1, 2)
    return keep, keep_edges

# ---- main loop ----

def open_image(zarr_path):
    with open(os.path.join(zarr_path, '0', 'zarr.json')) as f:
        meta = json.load(f)
    return tuple(int(s) for s in meta['shape']), np.dtype(meta['data_type'])

test_folder_names = sorted(d.replace('.zarr', '') for d in os.listdir(TEST_DIR) if d.endswith('.zarr'))

all_rows = []
for folder_name in test_folder_names:
    zarr_path = os.path.join(TEST_DIR, folder_name + '.zarr')
    shape, dtype = open_image(zarr_path)
    n_t = shape[0]

    root = zarr.open(zarr_path, mode='r')
    image = root['0']

    frames = []
    vols = []
    for t in range(n_t):
        vol = np.asarray(image[t])
        vols.append(vol)
        coords = detect_blobs(vol)
        if len(coords) > 0:
            coords = refine_centroids(vol, coords)
        frames.append(coords)

    node_ids, node_t, node_z, node_y, node_x, edges = link_twopass(frames, tight_um=6.0, loose_um=8.0)
    keep_ids, edges = filter_short_tracks(node_ids, edges, min_len=4)

    keep_mask = np.isin(node_ids, list(keep_ids))
    node_ids_f = node_ids[keep_mask]; node_t_f = node_t[keep_mask]
    node_z_f = node_z[keep_mask]; node_y_f = node_y[keep_mask]; node_x_f = node_x[keep_mask]

    for i in range(len(node_ids_f)):
        all_rows.append({
            'dataset': folder_name, 'row_type': 'node', 'node_id': int(node_ids_f[i]),
            't': int(node_t_f[i]), 'z': int(round(node_z_f[i])),
            'y': int(round(node_y_f[i])), 'x': int(round(node_x_f[i])),
            'source_id': -1, 'target_id': -1,
        })
    for s, t in edges:
        all_rows.append({
            'dataset': folder_name, 'row_type': 'edge', 'node_id': -1,
            't': -1, 'z': -1, 'y': -1, 'x': -1,
            'source_id': int(s), 'target_id': int(t),
        })

    print(f'{folder_name}: {len(node_ids_f)} nodes, {len(edges)} edges')

submission = pd.DataFrame(all_rows, columns=['dataset', 'row_type', 'node_id', 't', 'z', 'y', 'x', 'source_id', 'target_id'])
submission.index.name = 'id'
submission.to_csv('submission.csv')
print(f'Done. {len(submission)} rows written to submission.csv')

44b6_0113de3b: 24762 nodes, 23710 edges
44b6_0b24845f: 20336 nodes, 18036 edges
6bba_05b6850b: 5539 nodes, 5266 edges
6bba_05db0fb1: 54323 nodes, 50691 edges
Done. 202663 rows written to submission.csv
